# Wasserstein Ascend Descend
### (C. and N.) Garcia Trillos

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import numpy as np
from Robust_nn.WAD import WAD2scale
import matplotlib.pyplot as plt
from utils.utils import read_vision_dataset
from utils.convnet import ConvNet
from utils.convnet_silu import ConvNetSiLU
import os
import torch
import gc
import torch.nn as nn
from torch.optim.swa_utils import AveragedModel, SWALR


**Read the DataLoader**

In [4]:
trainloader, testloader = read_vision_dataset('../data', dataset='MNIST',batch_size = 1024)

In [5]:
# Create some networks

n_nets = 5
net_lst = [ConvNet() for i in range(n_nets)]
avg_nets =[ConvNet() for i in range(n_nets*4)]

adv_net = WAD2scale(net_list = net_lst, avg_nets = avg_nets, 
                    trainloader=trainloader, testloader = testloader, 
                    device = None , criterion= nn.CrossEntropyLoss(), 
                    scale_factor=5, num_adverse=2,
                    kappa = { 'param': 0.2, 'adv': 0.2   },
                    max_batches= 25)

In [6]:
adv_net.set_optimizer()
adv_net.train(epochs = 10)


Epoch: 0
tensor([0.5000, 0.5000])
tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000])
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|
Epoch: 1
tensor([0.5047, 0.4953])
tensor([0.2033, 0.1935, 0.1964, 0.2169, 0.1899], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|
Epoch: 2
tensor([0.5035, 0.4965])
tensor([0.2027, 0.1953, 0.1979, 0.2106, 0.1936], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|
Epoch: 3
tensor([0.5063, 0.4937])
tensor([0.2038, 0.1933, 0.1987, 0.2102, 0.1940], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|
Epoch: 4
tensor([0.5093, 0.4907])
tensor([0.2039, 0.1930, 0.1988, 0.2103, 0.1940], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|
Epoch: 5
tensor([0.5123, 0.4877])
tensor([0.2039, 0.1929, 0.1988, 0.2103, 0.1940], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|1

KeyboardInterrupt: 

In [7]:
adv_net.save_model('Test1_kappa0p2-3')

Saving...
done.


In [5]:
adv_net.set_optimizer()
adv_net.import_model('Test1_kappa0p2_2')

Loading...
done.


In [6]:
'''
Testing the model 

Peform the following tests:
1. Test for accuracy of the average  model
2. Train directly the model with the whole set of adversaries. Calculate...
3. Create directly the adversarial. 

SyntaxError: EOF while scanning triple-quoted string literal (4294075774.py, line 7)

In [24]:
adv_net.test_pgd(5)

epoch = 5, adv_acc = 17.49, clean_acc = 29.92
Saving the best model to ./checkpoint
Saving...
done.


(2.280136430263519, 17.49, 29.92)

In [29]:
adv_net.avg_adv_samples['original'][0].shape

torch.Size([512, 1, 28, 28])

**Creating and comparing results**

[tensor([0.1745, 0.3892, 0.6346, 0.0489, 0.0594]), tensor([0.0834, 0.4240, 0.5160, 0.3686, 0.5654]), tensor([0.6288, 0.8525, 0.3783, 0.9661, 0.8952])]
tensor([0.8867, 1.6657, 1.5288, 1.3836, 1.5201])


In [ ]:
# options = {'only o1':(True,False, False), 'only o2':(False, True, False), 'both': (True,True, False), 'none':(False,False,False), 'modO2':(False,True,True) } 
options = {'only o1':(True,False, False), 'both': (True,True, True), 'none':(False,False,False) } 
rvec = [2,4,np.inf]
deltav = [0.2]
mdict = {}
basepath = os.curdir
mpath = os.path.join(basepath,'models')

for i,(k,(o1,o2, mod_o2)) in enumerate(options.items()):
  auxd = {}
  for r in rvec:
    rstr = str(r)
    print('\n*******\n ** Case '+k+', r=',r,'\n ******')
    torch.manual_seed(0)
    np.random.seed(0)
    auxd[rstr] = {}
    network = ConvNet()
    net_RegTrin = OrdTwoL(network, trainloader, testloader,  device='cuda', delta =0.2, r= r, o2=o2, o1=o1, mod_o2=mod_o2) #'cuda'
    net_RegTrin.set_optimizer(optim_alg='Adam', args={'lr':1e-4})
    net_RegTrin.train(epochs=5, delta=deltav)
    auxd[rstr]['train_loss'] = net_RegTrin.train_loss.copy()
    auxd[rstr]['train_acc'] = net_RegTrin.train_acc.copy()
    auxd[rstr]['train_reg'] = net_RegTrin.train_reg.copy()
    auxd[rstr]['test_acc_adv'] = net_RegTrin.test_acc_adv.copy()
    auxd[rstr]['test_acc_clean'] = net_RegTrin.test_acc_clean.copy()
    auxd[rstr]['train_times'] = net_RegTrin.train_times.copy()
    torch.save(network.state_dict(), os.path.join(mpath, k+'_r_'+str(r)+'.pth' ))
    del(network)
    gc.collect()
    torch.cuda.empty_cache()
  mdict[k] = auxd
  with open('tests_all.txt','w') as data: 
      data.write(str(mdict))